# Component 3 - Phase 3: individual cell extraction (Sub-component 3.3)

StarDist cells (Phase 2 winner) -> complete cells, area / perimeter / compactness / **side count**.
Uses predictions already saved in Phase 2 - **CPU runtime is enough** (Runtime > Change runtime type > CPU).

In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Unit tests on synthetic tessellations (hexagons, random Voronoi, borders, holes, shapes)
!python -m pytest tests/test_extraction.py -q

In [ ]:
# 3. Step 3.6 - cell features: StarDist (out-of-fold) vs your reference masks, 40 crops
!python -m extraction.validate_crops

In [ ]:
# 4. Step 3.7 - extract cells on the 197 validation images, then compare with Konan
!python -m extraction.run_extract
!python -m extraction.run_extract --compare

In [ ]:
# 5. Figures: cells coloured by side count (one image per ECD band)
import pandas as pd
from IPython.display import Image, display
man = pd.read_csv("splits/manifest.csv")
val = man[(man["split"] == "val") & (man["status"] == "included")]
ids = [g.sample(1, random_state=0)["image_id"].iloc[0] for _, g in val.groupby("ecd_band")]
ARGS = " ".join('"' + i + '"' for i in ids)          # image IDs contain spaces
!python -m extraction.run_extract --figure {ARGS}
for i in ids:
    display(Image(f"{OUTPUT_ROOT}/extraction/full_val/stardist_finetune_flat/figures/{i}.png", width=1000))